# 智能文档分析师（Smart Document Analyst）

## 练习目标（理念）

多模态、AI 驱动的业务小应用：用户通过**对话式问答**上传并查询各类业务文档。

内置于 Jupyter + **Gradio UI**，可处理：

- 发票 / 合同 / 表单（常以图片出现）
- PDF、Word（`.docx`）
- 录音（Whisper 转写）
- 纯文本 / CSV / Markdown

## 和本课 Week 2 的关系

| 本课概念 | 本笔记本里你会看到 |
|----------|------------------|
| 多提供商客户端 | OpenAI / Anthropic / Gemini |
| System Prompt | `SYSTEM_PROMPT` 业务文档分析师人设 |
| 视觉（Vision） | 图片转 base64 塞进多模态消息 |
| Streaming | `stream_response` 对各家做增量 `yield` |
| Gradio multimodal | `ChatInterface` + 文件类型白名单 |

## 怎么跑

1. 先跑依赖安装格（`pip`）
2. 环境变量准备：`OPENAI_API_KEY`、`ANTHROPIC_API_KEY`、`GEMINI_API_KEY`
3. 自上而下运行初始化 / 引擎 / 文件处理 / UI
4. 最后一格 `demo.launch(...)`（含简单 `auth`）


In [ ]:
# ========== 依赖：安装/升级一次即可（Jupyter shell magic）==========

# -q 安静模式；-U 升级到较新版本；包名保持原样
!pip -q install -U gradio pillow openai anthropic google-genai numpy


In [ ]:
# ========== 导入 + 初始化三家 LLM 客户端 ==========

# 标准库：环境变量、字节流、JSON、计时（本格未必全用到，保持原导入）
import os
import io
import json
import time
# 类型标注：可读性 + IDE 提示；Generator 对应流式 yield
from typing import Optional, Dict, Any, Generator, List, Tuple

# Gradio：Web UI
import gradio as gr
# NumPy：数值/数组（本应用后续若处理音频数组会用到）
import numpy as np
# Pillow：打开与转换上传图片
from PIL import Image

# OpenAI 官方 SDK
from openai import OpenAI
# Anthropic 官方 SDK
from anthropic import Anthropic
# Google GenAI 新客户端 + types（拼 Content/Part）
from google import genai
from google.genai import types

# 客户端（假定已在您的环境中设置密钥）


# 分别从环境变量读取三家 Key（变量名必须与你的 .env 一致）
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')


# 只打印前缀做存在性检查；完整 key 不要打日志
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    # 原文末尾有空格与括号，保持原样不「修正」
    print("Anthropic API Key not set )")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    # 原文不完整的提示字符串，保持原样
    print("Google API Key not ")


# 默认构造三家客户端（密钥从各自环境变量约定读取）
openai = OpenAI()
anthropic = Anthropic()
# 注意：原代码 genai.Client() 前有两个空格，语义不变
gemini =  genai.Client()


In [ ]:
# ========== System Prompt + 模型注册表（UI 名 → provider / model_id）==========


# 业务文档分析师人设：整段英文影响模型行为，禁止翻译
SYSTEM_PROMPT = """You are an expert Business Document Analyst with deep expertise in:
- Invoices & receipts: extracting vendor details, line items, totals, tax, payment terms
- Contracts & agreements: identifying parties, key dates, obligations, risks, termination clauses
- Forms & applications: parsing fields, flagging missing or inconsistent data, summarizing intent
- General business documents: summarizing content, spotting anomalies, answering precise questions

Your behaviour:
- When a document image is first uploaded, automatically provide a structured overview:
  * Document type detected
  * Key parties or entities involved
  * Top 3-5 key findings or extracted fields
  * Any anomalies, missing info, or red flags 
- After the overview, invite the user to ask follow-up questions
- Answer questions precisely, referencing specific sections of the document
- When extracting structured data, return it as a clean markdown table
- For contract risk items use:  High Risk /  Medium Risk /  Low Risk
- Be concise, professional, and business-focused at all times
- If something in the document is unclear or illegible, say so honestly
"""

# 模型注册表 — 将 UI 显示名称映射到提供者 + 模型 ID
MODELS = {
    # OpenAI 多模态小模型：便宜、适合发票/截图问答
    "GPT-4o mini (OpenAI)": {
        "provider": "openai",
        "model_id": "gpt-4o-mini"
    },
    # Anthropic Claude Sonnet：长文/合同分析常用
    "Claude Sonnet (Anthropic)": {
        "provider": "anthropic",
        "model_id": "claude-sonnet-4-5"
    },
    # Google Gemini Flash：偏速度的多模态选项
    "Gemini 2.5 Flash (Google)": {
        "provider": "gemini",
        "model_id": "gemini-2.5-flash"
    }
}


In [ ]:
# ========== 图像工具：PIL → base64（给 vision 消息用）==========


# base64：把二进制编码成可塞进 JSON/URL 的文本
import base64
# BytesIO：内存中的「文件」，避免先落地再读
from io import BytesIO

def pil_to_base64(pil_image: Image.Image, format: str = "JPEG") -> str:
    """把 PIL 图像编码为 base64 字符串；RGBA/调色板模式先转 RGB。"""
    # 带透明通道或调色板的图，JPEG 不支持 → 先转 RGB
    if pil_image.mode in ("RGBA", "P", "LA"):
        pil_image = pil_image.convert("RGB")
    
    # 写入内存缓冲区
    buffer = BytesIO()
    pil_image.save(buffer, format=format)
    # 指针回到开头，准备读取字节
    buffer.seek(0)
    # b64encode 得到 bytes，再 decode 成 utf-8 字符串
    return base64.b64encode(buffer.getvalue()).decode("utf-8")


# --- 快速测试 ---
# 创建一个小虚拟图像并验证功能是否有效
# 纯红 100x100，不依赖真实上传文件
test_img = Image.new("RGB", (100, 100), color=(255, 0, 0))  # solid red square
# 跑一遍编码；成功则得到非空字符串（本格未 print，可自行加）
test_b64 = pil_to_base64(test_img)


In [ ]:
# ========== 流式推理引擎：按 provider 分支拼消息并 yield 累计文本 ==========


def stream_response(
    message: str,
    history: List[Dict],
    model_key: str,
    image: Optional[Image.Image] = None
) -> Generator[str, None, None]:
    """按所选模型流式生成回复；有图则走各家多模态消息格式。"""

    # 从注册表取出 provider 与真实 model_id
    model_cfg = MODELS[model_key]
    provider  = model_cfg["provider"]
    model_id  = model_cfg["model_id"]
    # 有图才编码；无图则 b64=None
    b64       = pil_to_base64(image) if image else None

    # ------------------------------------------------------------------ #
    # OpenAI 分支
    # ------------------------------------------------------------------ #
    if provider == "openai":

        # 以 OpenAI 格式构建消息历史记录：先 system
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]

        # 添加历史记录中的先前回合（role/content 字典）
        for turn in history:
            messages.append({"role": turn["role"], "content": turn["content"]})

        # 构建当前用户消息 - 包含图像（如果存在）
        if b64:
            # 多模态 content：image_url + text
            user_content = [
                {"type": "image_url",
                 "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
                {"type": "text", "text": message}
            ]
        else:
            # 纯文本时 content 直接是字符串
            user_content = message

        messages.append({"role": "user", "content": user_content})

        # 来自 OpenAI 的流：stream=True
        stream = openai.chat.completions.create(
            model=model_id,
            messages=messages,
            stream=True
        )

        accumulated = ""
        for chunk in stream:
            # delta.content 可能为 None（如 role 块）
            delta = chunk.choices[0].delta.content
            if delta:
                accumulated += delta
                yield accumulated  # yield full string so far (Gradio requirement)

    # ------------------------------------------------------------------ #
    # Anthropic 分支
    # ------------------------------------------------------------------ #
    elif provider == "anthropic":

        # 以 Anthropic 格式构建消息历史记录（system 单独参数，不进 messages）
        messages = []

        for turn in history:
            messages.append({"role": turn["role"], "content": turn["content"]})

        # 构建当前用户消息 - 包含图像（如果存在）
        if b64:
            user_content = [
                {"type": "image",
                 "source": {
                     "type": "base64",
                     "media_type": "image/jpeg",
                     "data": b64
                 }},
                {"type": "text", "text": message}
            ]
        else:
            # Anthropic 即使纯文本也常用 list-of-blocks
            user_content = [{"type": "text", "text": message}]

        messages.append({"role": "user", "content": user_content})

        # 来自 Anthropic 的流媒体：上下文管理器 + text_stream
        with anthropic.messages.stream(
            model=model_id,
            max_tokens=2048,
            system=SYSTEM_PROMPT,
            messages=messages
        ) as stream:
            accumulated = ""
            for text_chunk in stream.text_stream:
                accumulated += text_chunk
                yield accumulated

    # ------------------------------------------------------------------ #
    # Gemini 分支
    # ------------------------------------------------------------------ #
    elif provider == "gemini":

        # 构建内容列表——Gemini 使用角色/部分字典的平面列表
        contents = []

        for turn in history:
            # Gemini 用 “model” 代替 “assistant”
            role = "model" if turn["role"] == "assistant" else "user"
            contents.append(
                types.Content(
                    role=role,
                    parts=[types.Part.from_text(text=turn["content"])]
                )
            )

        # 构建当前用户回合 - 包括图像（如果存在）
        if b64:
            # Gemini 要原始 bytes，所以先 decode base64
            image_bytes = base64.b64decode(b64)
            current_parts = [
                types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg"),
                types.Part.from_text(text=message)
            ]
        else:
            current_parts = [types.Part.from_text(text=message)]

        contents.append(types.Content(role="user", parts=current_parts))

        # 来自 Gemini 的直播；system_instruction 单独配置
        stream = gemini.models.generate_content_stream(
            model=model_id,
            contents=contents,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                max_output_tokens=2048,
            )
        )

        accumulated = ""
        for chunk in stream:
            # 有的 chunk 可能没有 text
            if chunk.text:
                accumulated += chunk.text
                yield accumulated


In [ ]:
# ========== 文件处理：按扩展名分流 → 统一 dict 结构 ==========

# pathlib：用 Path 取后缀与文件名
import pathlib

# 安装所需的库（如果尚不存在）：运行时 pip，输出吞掉
import subprocess
subprocess.run(["pip", "install", "pypdf", "python-docx", "-q"], 
               capture_output=True)

# PDF / Word 解析库
from pypdf import PdfReader
from docx import Document as DocxDocument


def process_uploaded_file(file_path: str) -> Dict[str, Any]:
    """按扩展名处理上传文件，返回 type/content/image/name 规范化字典。"""
    path = pathlib.Path(file_path)
    # 小写后缀，避免 .PDF / .Pdf 漏判
    ext  = path.suffix.lower()
    name = path.name

    # ── 图片：交给 vision，不在这里 OCR ──────────────────────────────
    if ext in [".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff", ".gif"]:
        img = Image.open(file_path).convert("RGB")
        return {"type": "image", "image": img, "content": None, "name": name}

    # ── PDF：逐页 extract_text，拼成带页码标记的长文本 ───────────────
    elif ext == ".pdf":
        try:
            reader = PdfReader(file_path)
            pages  = []
            for i, page in enumerate(reader.pages):
                text = page.extract_text()
                if text and text.strip():
                    pages.append(f"[Page {i+1}]\n{text.strip()}")
            # 扫描件可能抽不出字：保留原警告文案
            full_text = "\n\n".join(pages) if pages else "⚠️ No extractable text found in PDF (may be scanned image)."
            return {"type": "pdf", "content": full_text, "image": None, "name": name}
        except Exception as e:
            return {"type": "error", "content": f"PDF read error: {e}", "image": None, "name": name}

    # ── DOCX：拼非空段落 ─────────────────────────────────────────────
    elif ext in [".docx", ".doc"]:
        try:
            doc        = DocxDocument(file_path)
            paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]
            full_text  = "\n\n".join(paragraphs) if paragraphs else "⚠️ No text found in document."
            return {"type": "docx", "content": full_text, "image": None, "name": name}
        except Exception as e:
            return {"type": "error", "content": f"DOCX read error: {e}", "image": None, "name": name}

    # ── 音频：OpenAI Whisper 转写后当文本注入 ────────────────────────
    elif ext in [".mp3", ".wav", ".m4a", ".ogg", ".flac", ".webm"]:
        try:
            with open(file_path, "rb") as audio_file:
                transcript = openai.audio.transcriptions.create(
                    model="whisper-1",
                    file=audio_file,
                    response_format="text"
                )
            # SDK 可能直接返回 str，也可能是带 .text 的对象
            transcribed = transcript if isinstance(transcript, str) else transcript.text
            full_text   = f"[Audio Transcription — {name}]\n\n{transcribed}"
            return {"type": "audio", "content": full_text, "image": None, "name": name}
        except Exception as e:
            return {"type": "error", "content": f"Audio transcription error: {e}", "image": None, "name": name}

    # ── 纯文本 / CSV / Markdown：直接读文件 ──────────────────────────
    elif ext in [".txt", ".csv", ".md"]:
        try:
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                full_text = f.read()
            return {"type": "text", "content": full_text, "image": None, "name": name}
        except Exception as e:
            return {"type": "error", "content": f"Text read error: {e}", "image": None, "name": name}

    # ── 不支持：返回 error 类型，由上层 yield 给用户 ─────────────────
    else:
        return {
            "type": "error",
            "content": f"⚠️ Unsupported file type: {ext}. Supported: images, PDF, DOCX, audio, TXT, CSV",
            "image": None,
            "name": name
        }


In [ ]:
# ========== Gradio UI：多文件类型聊天入口 + 模型下拉 + 启动 ==========


def chat(
    message: Dict,
    history: List[Dict],
    model_key: str
) -> Generator[str, None, None]:
    """多模态聊天处理：图片走 vision；其它类型抽文本注入 prompt。"""

    # 用户打字内容；可能为空（只上传文件）
    user_text  = message.get("text", "").strip()
    image      = None
    extra_text = ""   # will hold extracted content from non-image files

    # --- 处理上传的文件（如果有）---
    files = message.get("files", [])
    if files:
        # Gradio 有时给 dict（含 path），有时直接给路径字符串
        file_path = files[0]["path"] if isinstance(files[0], dict) else files[0]
        result    = process_uploaded_file(file_path)

        if result["type"] == "image":
            # 视觉通路——将 PIL 图像传递给 LLM
            image = result["image"]

        elif result["type"] == "error":
            # 直接把错误信息流式（一次）返回，结束本轮
            yield result["content"]
            return

        else:
            # 文本路径 - 将提取的内容添加到提示中
            doc_type_label = {
                "pdf"  : "📄 PDF Document",
                "docx" : "📝 Word Document",
                "audio": "🎙️ Audio Transcript",
                "text" : "📃 Text File"
            }.get(result["type"], "Document")

            # 英文包装文本会进模型 prompt，保持原样
            extra_text = (
                f"The user has uploaded a {doc_type_label} named '{result['name']}'.\n"
                f"Here is its content:\n\n"
                f"{'='*60}\n"
                f"{result['content']}\n"
                f"{'='*60}\n\n"
            )

    # --- 至少需要一些输入 ---
    if not user_text and not extra_text and image is None:
        yield "⚠️ Please upload a document and/or type a question."
        return

    # --- 构建最终提示 ---
    if extra_text and not user_text:
        # 文件上传无问题 → 自动分析
        final_prompt = extra_text + "Please analyze this document and provide a structured overview."
    elif extra_text and user_text:
        # 文件+问题一起
        final_prompt = extra_text + f"User question: {user_text}"
    else:
        # 仅文本或仅图像
        final_prompt = user_text or "Please analyze this document and provide a structured overview."

    # --- 流响应：把累计字符串继续 yield 给 ChatInterface ---
    for chunk in stream_response(
        message=final_prompt,
        history=history,
        model_key=model_key,
        image=image
    ):
        yield chunk


# ------------------------------------------------------------------ #
# 用户界面布局
# ------------------------------------------------------------------ #
with gr.Blocks(title="Smart Document Analyst") as demo:

    # 界面说明 Markdown（展示文案保持原样）
    gr.Markdown("""
    # 智能文档分析师
    **Upload any business document** and ask questions about it.  
    Supports:  Images ·  PDF ·  DOCX · 🎙️ Audio ·  TXT/CSV  
    Multi-turn Q&A · Switch models anytime to compare responses.
    """)

    # 模型下拉：choices 来自 MODELS.keys()；默认 Claude
    model_selector = gr.Dropdown(
        choices=list(MODELS.keys()),
        value="Claude Sonnet (Anthropic)",
        label="🤖 Select AI Model",
        interactive=True
    )

    # multimodal=True：文本框可带文件；additional_inputs 传入模型选择
    gr.ChatInterface(
        fn=chat,
    
        multimodal=True,
        additional_inputs=[model_selector],
        chatbot=gr.Chatbot(
            label="Document Q&A",
            height=520,
            placeholder=(
                " Upload a document using the **paperclip icon**, then ask:\n\n"
                "- *What is the total amount due?*\n"
                "- *Who are the parties in this contract?*\n"
                "- *Summarize the key clauses*\n"
                "- *Extract all line items as a table*\n"
                "- *Are there any red flags?*"
            )
        ),
        textbox=gr.MultimodalTextbox(
            placeholder="Upload a document and/or type your question...",
            file_types=[
                ".jpg", ".jpeg", ".png", ".webp", ".bmp",   
                ".pdf",                                       
                ".docx", ".doc",                             
                ".mp3", ".wav", ".m4a", ".ogg", ".webm",    
                ".txt", ".csv", ".md"                        
            ],
            file_count="single"
        ),
    )

    gr.Markdown("""
    ---
    💡 **Tips:** Switch models mid-conversation to compare analysis.  
    Audio files are auto-transcribed before analysis.  
    For scanned PDFs (image-based), upload as image file instead.
    """)

# 启动：Soft 主题；share 公网链接；简单双用户 auth；防止线程锁死 notebook
demo.launch(
    theme=gr.themes.Soft(),
    debug=False,
    share=True,                         
    auth=[                               
        ("admin", "admin123"),
        ("analyst", "docs2024"),
    ],
    auth_message="🔐 Please log in to access the Smart Document Analyst",
     prevent_thread_lock=True  
)
